In [11]:
from glob import glob
import pandas as pd
from scipy import stats
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.python.ops import rnn, rnn_cell
import numpy as np
from sklearn.model_selection import train_test_split
%matplotlib inline
plt.style.use('ggplot')


In [12]:
base_dir = "audios/model/train/"

sound_file_paths = [os.path.basename(file)
                    for file in glob(f'{base_dir}*.wav')]
# Output tags
sound_names = [file[:3] for file in sound_file_paths]

In [13]:
def feature_normalize(dataset):
    mu = np.mean(dataset, axis=0)
    sigma = np.std(dataset, axis=0)
    return (dataset - mu) / sigma


def windows(data, window_size):
    start = 0
    while start < len(data):
        yield int(start), int(start + window_size)
        start += (window_size / 2)  # stepping at half window size


def extract_features(base_dir, sound_file_paths, sound_names, bands=20, frames=41):
    window_size = 512 * (frames - 1)
    mfccs = []
    labels = []
    for i, sound_file_path in enumerate(sound_file_paths):
        sound_file_full_path = os.path.join(base_dir, sound_file_path)
        sound_clip, s = librosa.load(sound_file_full_path)
        sound_clip = feature_normalize(sound_clip)
        label = sound_names[i]
        for (start, end) in windows(sound_clip, window_size):
            if(len(sound_clip[start:end]) == window_size):
                signal = sound_clip[start:end]
                # y: audio time series, sr: sampling rate, n_mfcc: number of MFCCs to return
                # librosa.feature.mfcc() function return numpy array with shape (bands, frames)
                # transpose since the model expects time axis(frames) come first
                mfcc = librosa.feature.mfcc(y=signal, sr=s, n_mfcc=bands).T
                mfccs.append(mfcc)
                labels.append(label)
    features = np.asarray(mfccs)
    return np.array(features), np.array(labels, dtype=np.str)


def one_hot_encode(labels):
    return np.asarray(pd.get_dummies(labels), dtype=np.float32)


In [14]:
bands = 20
frames = 41
features, labels = extract_features(
    base_dir, sound_file_paths, sound_names, bands=bands, frames=frames)
labels = one_hot_encode(labels)


C:\Users\tomsb\AppData\Local\Temp\ipykernel_6348\2331868213.py:33: DeprecationWarning: `np.str` is a deprecated alias for the builtin `str`. To silence this warning, use `str` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.str_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  return np.array(features), np.array(labels, dtype=np.str)


In [15]:
features.shape


(479, 41, 20)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=10)


In [17]:
learning_rate = 0.01
training_iters = 300
batch_size = 5
display_step = 100

# Network Parameters
n_input = bands
n_steps = frames
n_hidden = 64
n_classes = 2  # pos, neg


In [18]:
# Define nodes
inputs = tf.keras.Input(shape=(n_steps, n_input))
x = tf.keras.layers.Dense(n_hidden, activation=tf.nn.relu)(inputs)
y = tf.keras.Input(shape=(n_classes,))
keep_prob = tf.keras.Input(shape=(), dtype=tf.float32, name='keep_prob')

# Define weights and biases
weight = tf.Variable(tf.random.normal([n_hidden, n_classes]))
bias = tf.Variable(tf.random.normal([n_classes]))


In [19]:
# Define LSTM cell
lstm_cell = tf.keras.layers.LSTMCell(n_hidden)

# Define RNN layer
rnn = tf.keras.layers.RNN(lstm_cell, return_sequences=True)

# Define output layer
outputs = tf.keras.layers.Dense(n_classes, activation='softmax')(rnn(x))

# Define loss and optimizer
y_one_hot = tf.one_hot(tf.argmax(outputs, axis=-1), n_classes)
loss_f = tf.keras.losses.sparse_categorical_crossentropy(y, outputs)
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

# Evaluate model
correct_pred = tf.keras.metrics.categorical_accuracy(y_one_hot, outputs)


In [20]:
# Compile model
model = tf.keras.Model(inputs=[x, keep_prob], outputs=[outputs])
model.compile(optimizer=optimizer, loss=loss_f, metrics=[correct_pred])

# Train model
model.fit(x=X_train, y=y_train, epochs=training_iters, 
          batch_size=batch_size, verbose=1, validation_data=(X_test, y_test))

# Save model
model.save('./model/acoustic.h5')

# Predict
test_features = extract_features('./data', ["test_new_60Hz.wav"], ["Unknown"])
y_predicts = model.predict(test_features, batch_size=batch_size, verbose=1)

# Get predicted label
predicted_logit = tf.argmax(tf.convert_to_tensor(y_predicts), 1)
predicted_label = LABELS[predicted_logit]
predicted_probability = stats.mode(np.argmax(y_predicts, 1))[
    1][0] / len(y_predicts)


TypeError: Keras symbolic inputs/outputs do not implement `__len__`. You may be trying to pass Keras symbolic inputs/outputs to a TF API that does not register dispatching, preventing Keras from automatically converting the API call to a lambda layer in the Functional Model. This error will also get raised if you try asserting a symbolic input/output directly.